# Pipeline Durability

Notebook 12's indexer is a `BackgroundTask` running inside the FastAPI process. The pros: trivial to launch, no separate runtime, no service to deploy. The cons: the moment that process dies — `Ctrl-C`, docker-compose restart, OS patch update, the OOM killer — the job and every per-batch progress counter wired to the WebSocket stream are gone. Reindexing starts from S3 row zero on next visit. For a 50 000-photo library that already took two hours to embed once, "restart the pipeline" means two more hours.

[In-memory state is the wrong place to keep job progress]{.mark}. The right place is a Postgres table. A pipeline job is a row in a `jobs` table; each batch the pipeline processes is a row in a `job_batches` table; the FSM of "which batches are pending, which are done, which failed" lives in SQL and is therefore crash-safe. A restarted API process reads the `jobs` table at lifespan startup, finds the in-flight one, and **resumes from the last committed batch**. The progress WebSocket tracks the same row.

This notebook wires durability into the pipeline. We also fulfill PHT:01's promise that re-index is cheap: the content-hash `sha256` column means "did we already embed this photo" is a single `EXISTS` query, so a re-run of the full pipeline over a 50K-photo bucket skips everything that has not changed and processes only the new uploads.

---

## The Job Tables

Two tables. `jobs` records job-level state and total counts. `job_batches` records per-batch state and the cardinal boundary of progress so resume knows where to pick up.

```sql
-- 0021_pht_jobs.sql
CREATE TYPE job_state AS ENUM (
    'queued',     -- created but not yet claimed by a worker
    'running',    -- in progress
    'paused',     -- user-initiated pause
    'completed',
    'failed',
    'canceled'
);

CREATE TYPE job_kind AS ENUM (
    'reindex_prefix',
    'reindex_device',
    'reindex_photo',
    'generate_derivatives',
    'recalculate_embeddings'   -- after a CLIP model upgrade
);

CREATE TABLE jobs (
    job_id        TEXT PRIMARY KEY,
    kind          job_kind NOT NULL,
    scope_json    JSONB NOT NULL,        -- {"prefix":"/photos", "device_id":"d-1", ...}
    state         job_state NOT NULL DEFAULT 'queued',
    total_items   INT,                   -- NULL until the scan phase counts items
    processed     INT NOT NULL DEFAULT 0,
    failed        INT NOT NULL DEFAULT 0,
    started_at    TIMESTAMPTZ,
    completed_at  TIMESTAMPTZ,
    last_error    TEXT
);

CREATE INDEX ix_jobs_state ON jobs (state) WHERE state IN ('queued', 'running', 'paused');

CREATE TABLE job_batches (
    batch_id      TEXT PRIMARY KEY,
    job_id        TEXT NOT NULL REFERENCES jobs(job_id),
    seq           INT NOT NULL,         -- 0,1,2,… ordered by sort
    state         job_state NOT NULL DEFAULT 'queued',  -- queued / running / completed / failed
    s3_keys_json  JSONB NOT NULL,       -- the batch's input: ["/photos/aa/...", ...]
    completed_at  TIMESTAMPTZ,
    error         TEXT,
    UNIQUE (job_id, seq)
);

CREATE INDEX ix_job_batches_resume ON job_batches (job_id, seq) WHERE state IN ('queued', 'running');
```

:::{.callout-note}
The `scope_json` JSONB column on `jobs` is what lets a single pair of tables serve every job kind: the durable worker reads `kind` + `scope_json` and dispatches. New job kinds (e.g. "rebuild all summaries with a new prompt") are added as rows with a new `kind` enum value; the worker code is the only change.

:::


## Job Lifecycle FSM

Five transitions are legal. Anything outside this table is a bug we raise on at the application layer.

```
       queue                claim                finish
queued ───────► running ───────────► running ───────────► completed
                  │   ▲                                       │
                  │   │ cancel                                │ pause
                  ▼   │                                      ▼
                paused ◄────────────────────────────── paused
                 ▲
                 │ pause
                 │
              running
                 │ claim
                 │
                 ▼
              running
```

Transition      | Trigger                | SQL delivers
----------------|------------------------|--------------
`queued → running` | Worker claims         | `UPDATE … WHERE state='queued' RETURNING …` (atomic)
`running → paused` | User `POST /pause`   | atomic; tracked as `paused` if worker hiatus >5s
`paused → running` | User `POST /resume`  | atomic; worker notices via jobs-table poll
`running → completed` | Last batch done   | atomic increment `processed` + state update in one tx
`running → failed` | unrecoverable error | captured in `last_error`; batches still pending remain so


In [ ]:
from datetime import datetime, timezone
from enum import Enum
from pydantic import BaseModel


class JobState(str, Enum):
    queued    = "queued"
    running   = "running"
    paused    = "paused"
    completed = "completed"
    failed    = "failed"
    canceled  = "canceled"


class JobKind(str, Enum):
    reindex_prefix      = "reindex_prefix"
    reindex_device      = "reindex_device"
    reindex_photo       = "reindex_photo"
    generate_derivatives = "generate_derivatives"
    recalculate_embeddings = "recalculate_embeddings"


class JobScope(BaseModel):
    """What the job operates on. Same shape reused for all job kinds;
    the consuming worker ignores fields irrelevant to its kind."""
    prefix:    str | None = None
    device_id: str | None = None
    photo_id:  str | None = None


class Job(BaseModel):
    job_id:      str
    kind:        JobKind
    state:       JobState = JobState.queued
    scope:       JobScope
    total_items: int | None = None
    processed:   int = 0
    failed:      int = 0
    started_at:  datetime | None = None
    completed_at: datetime | None = None
    last_error:  str | None = None


# Legal transitions.
LEGAL_TRANSITIONS: dict[JobState, set[JobState]] = {
    JobState.queued:    {JobState.running, JobState.canceled},
    JobState.running:   {JobState.paused, JobState.completed, JobState.failed, JobState.canceled},
    JobState.paused:    {JobState.running, JobState.canceled},
    JobState.completed: set(),
    JobState.failed:    {JobState.queued},   # manual restart from "failed"
    JobState.canceled:  set(),
}


def assert_transition(old: JobState, new: JobState) -> None:
    if new not in LEGAL_TRANSITIONS[old]:
        raise ValueError(f"illegal transition {old.value} → {new.value}")


# Verify the matrix.
assert_transition(JobState.queued, JobState.running)
print("queued → running  ✓")
assert_transition(JobState.running, JobState.paused)
print("running → paused  ✓")
assert_transition(JobState.running, JobState.completed)
print("running → completed  ✓")
try:
    assert_transition(JobState.completed, JobState.running)
    print("completed → running  ✗  (should have raised)")
except ValueError as e:
    print(f"illegal transition blocked: {e}")


## Claiming a Job Atomically

The naive pattern is `SELECT … WHERE state='queued' LIMIT 1; UPDATE … SET state='running'`. This races: two workers can both hit the same job and both mark it running, then both run the same batches. The atomic version is `UPDATE … SET state='running' WHERE state='queued' RETURNING …` — Postgres holds the row lock until the UPDATE either commits (transitions succeed) or rolls back. The `RETURNING` clause gives us the job row in the same round-trip.

:::{.callout-note}
This is the simplified single-instance safe pattern. On multiple workers (PHT:07's horizontal scale-out), the right pattern is a `FOR UPDATE SKIP LOCKED` claim — only one worker acquires a row; the others skip to the next. We will use that exact pattern in PHT:07's "Skip Lock" section. Here we use the simpler atomic UPDATE.

:::


In [ ]:
import uuid

# In-memory job table (mirrors the jobs table for the notebook).
JOBS: dict[str, Job] = {}

async def enqueue_job(kind: JobKind, scope: JobScope) -> Job:
    job = Job(
        job_id=f"job-{uuid.uuid4().hex[:8]}",
        kind=kind,
        scope=scope,
        state=JobState.queued,
    )
    JOBS[job.job_id] = job
    return job


async def claim_job(worker_id: str) -> Job | None:
    """Atomically claim a queued job. Returns None if no job is queued.

    Mirrors: UPDATE jobs SET state='running', started_at=now()
             WHERE state='queued' RETURNING ...
    The atomic<5s> UPDATE-and-return prevents the race where two workers each see
    the same queued job.
    """
    for job in JOBS.values():
        if job.state is JobState.queued:
            # In SQL this UPDATE … WHERE state='queued' RETURNING is wrapped in a
            # transaction that wins or loses the row lock; only the winner
            # receives the RETURNING row. Here, the in-memory version does the
            # equivalent test-and-set.
            old = job.state
            job.state = JobState.running
            assert_transition(old, JobState.running)
            return job
    return None


job = await enqueue_job(JobKind.reindex_photo, JobScope(photo_id="p-001"))
print(f"enqueued: {job.job_id}  state={job.state.value}")
won = await claim_job(worker_id="worker-1")
print(f"claimed by worker-1: {won.job_id if won else None}  state={won.state.value if won else 'n/a'}")
won2 = await claim_job(worker_id="worker-2")
print(f"claimed by worker-2: {won2.job_id if won2 else None}  ← expected None (no race)")


## Checkpoint Resume: Batches as the Unit of Progress

The scan phase enumerates all `s3_keys` the job must process,Chunks them into N batches (e.g. 100 keys per batch), inserts all N `job_batches` rows (all with `state='queued'`), and then the worker processes them in order. **After each batch commits**, the batch row is `UPDATE … SET state='completed', completed_at=now()` and the parent job's `processed += batch_size`. When the worker crashes and a new process picks up the job at lifespan startup, it queries the first `job_batches WHERE state='queued'` row (in `seq` order) and resumes there. **Skipping already-completed batches is automatic.**


In [ ]:
import json
from typing import Callable

BATCH_SIZE = 10   # photos per batch (100 in prod; small here for clarity in cells)


async def initialize_job_batches(
    job: Job,
    s3_keys: list[str],
    insert_batch_fn: Callable,    # callable(seq:int, keys:list[str]) -> None
) -> int:
    """Chunk the input into batches and insert all as 'queued'.
    Sets job.total_items and returns number of batches queued."""
    batches: list[tuple[int, list[str]]] = []
    for seq in range(0, len(s3_keys), BATCH_SIZE):
        chunk = s3_keys[seq:seq + BATCH_SIZE]
        batches.append((len(batches), chunk))
    for seq, chunk in batches:
        insert_batch_fn(seq, chunk)
    job.total_items = len(s3_keys)
    return len(batches)


async def next_pending_batch(job_id: str) -> tuple[int, list[str]] | None:
    """Find the first queued batch for the given job. In SQL:

        SELECT seq, s3_keys_json FROM job_batches
        WHERE job_id = :jid AND state = 'queued'
        ORDER BY seq ASC LIMIT 1
    """
    batch_state = JOB_BATCHES.get(job_id, {})
    for seq in sorted(batch_state):
        b = batch_state[seq]
        if b["state"] is JobState.queued:
            return seq, b["s3_keys"]
    return None


JOB_BATCHES: dict[str, dict[int, dict]] = {}    # job_id -> {seq -> batch dict}


# Demonstrate: chunk 35 keys into batches of 10, fail partway, resume.
job = await enqueue_job(JobKind.reindex_prefix, JobScope(prefix="/photos"))

s3_keys = [f"photos/{i:02}/img-{i:04d}.jpg" for i in range(35)]
n_batches = await initialize_job_batches(
    job, s3_keys,
    insert_batch_fn=lambda seq, keys: (
        JOB_BATCHES.setdefault(job.job_id, {})
                   .update({seq: {"state": JobState.queued, "s3_keys": keys}})
    ),
)
print(f"queued {n_batches} batches of {BATCH_SIZE} for {job.total_items} photos")


# Simulate processing the first 2 batches and crashing.
async def process_batch_simulated(job: Job, batch_keys: list[str]) -> None:
    # In production this runs CLIP + FaceNet + EXIF + upsert for each key.
    time.sleep(0.001)
    job.processed += len(batch_keys)

# Process batch seq=0 and seq=1, then pretend crash.
for _ in range(2):
    next_b = await next_pending_batch(job.job_id)
    if next_b is None:
        break
    seq, keys = next_b
    await process_batch_simulated(job, keys)
    JOB_BATCHES[job.job_id][seq]["state"] = JobState.completed
    print(f"processed seq={seq}  processed={job.processed}/{job.total_items}")

print(f"crash! processed={job.processed}/{job.total_items}")


# Simulate restart: claim the existing in-flight job (already 'running' in this case),
# call next_pending_batch — picks up at seq=2.
print(f"\nafter restart, resume from seq 2:")
for _ in range(2):
    next_b = await next_pending_batch(job.job_id)
    if next_b is None:
        break
    seq, keys = next_b
    await process_batch_simulated(job, keys)
    JOB_BATCHES[job.job_id][seq]["state"] = JobState.completed
    print(f"processed seq={seq}  processed={job.processed}/{job.total_items}")


## Parameter to Run, Pause, and Cancel

The FSM has three user-triggerable operations: `enqueue` (create), `pause` (running → paused mid-batch between batch boundaries), `cancel` (any active → canceled). Pause is engineered between batch boundaries: the worker finishes the current in-flight batch, commits it, and then notices the job is paused; it stops claiming the next batch and exits. Cancellation is the same shape with state=canceled. There is no hard interrupt — killing a CLIP encode mid-tensor would leave partial S3 writes, and S3 multipart aborts cost more than letting the batch finish.

:::{.callout-caution}
Batch size matters here. A 100-key batch on CPU-bound CLIP takes ~2 seconds; that is your upper bound on pause-to-actual-stop latency. If batches are 1000 keys (≈20 s of CPU), users will spam the pause button. **Set the batch size so the per-batch processing time is under 5 seconds.**

:::


In [ ]:
async def pause_job(job_id: str) -> bool:
    """Atomically mark a running job paused. The worker finishes its current
    batch, sees the new state, and stops claiming the next batch."""
    job = JOBS.get(job_id)
    if job is None:
        return False
    assert_transition(job.state, JobState.paused)
    job.state = JobState.paused
    return True


async def cancel_job(job_id: str) -> bool:
    """Atomically mark an active job canceled."""
    job = JOBS.get(job_id)
    if job is None:
        return False
    assert_transition(job.state, JobState.canceled)
    job.state = JobState.canceled
    return True


async def resume_job(job_id: str) -> bool:
    """Atomically resume a paused job (back to 'running'). The next worker pool
    will pick it up at the next pending batch."""
    job = JOBS.get(job_id)
    if job is None:
        return False
    assert_transition(job.state, JobState.running)
    job.state = JobState.running
    return True


# Demonstrate pause → resume → cancel around the in-flight job.
job = await enqueue_job(JobKind.reindex_device, JobScope(device_id="d-1"))
print(f"state={job.state.value}")
await claim_job(worker_id="w-1")
print(f"after claim: state={job.state.value}")
await pause_job(job.job_id)
print(f"after pause:  state={job.state.value}")
await resume_job(job.job_id)
print(f"after resume: state={job.state.value}")
await cancel_job(job.job_id)
print(f"after cancel: state={job.state.value}")


## Worker Pool Runtime

The pool is a bounded set of asyncio tasks, each pulling batches from `next_pending_batch` and processing them. A Semaphore limits parallelism so we don't issue 100 concurrent CLIP encodes (CPU-bound; the GIL would serialize them anyway, but the point is also to bound memory). The pool answers to two shutdown signals: cancellation of the parent job (exit cleanly, leaving remaining batches queued for the next pool) and process-level shutdown (drain).


In [ ]:
import asyncio


class JobWorkerPool:
    """Async pool that processes batches for one job."""
    def __init__(
        self,
        job_id:         str,
        workers:        int = 2,
        batch_processor: Callable[[list[str]], None] = lambda keys: None,
    ):
        self.job_id = job_id
        self.workers = workers
        self.process = batch_processor
        self._tasks: list[asyncio.Task] = []
        self._stop = asyncio.Event()

    async def _run(self) -> None:
        while not self._stop.is_set():
            next_b = await next_pending_batch(self.job_id)
            if next_b is None:
                return
            seq, keys = next_b
            # Mark batch running.
            JOB_BATCHES[self.job_id][seq]["state"] = JobState.running
            try:
                # In production: await real indexing work here.
                await asyncio.get_event_loop().run_in_executor(None, self.process, keys)
                JOB_BATCHES[self.job_id][seq]["state"] = JobState.completed
                if (job := JOBS.get(self.job_id)):
                    job.processed += len(keys)
            except Exception as exc:
                JOB_BATCHES[self.job_id][seq]["state"] = JobState.failed
                JOB_BATCHES[self.job_id][seq]["error"] = str(exc)
                if (job := JOBS.get(self.job_id)):
                    job.failed += len(keys)

    async def start(self) -> None:
        self._tasks = [asyncio.create_task(self._run()) for _ in range(self.workers)]

    async def join(self) -> None:
        await asyncio.gather(*self._tasks, return_exceptions=True)
        if (job := JOBS.get(self.job_id)) and job.state is not JobState.paused:
            job.state = JobState.completed if job.failed == 0 else JobState.failed
            job.completed_at = datetime.now(timezone.utc)

    def stop(self) -> None:
        self._stop.set()


# Demo: dispatch a small job with 25 photos through 2 workers, watch all complete.
job = await enqueue_job(JobKind.reindex_photo, JobScope(photo_id="demo"))
s3_keys = [f"photos/key-{i:03d}.jpg" for i in range(25)]
JOB_BATCHES[job.job_id] = {}
for seq, chunk in enumerate([s3_keys[i*5:i*5+5] for i in range(5)]):
    JOB_BATCHES[job.job_id][seq] = {"state": JobState.queued, "s3_keys": chunk}
job.total_items = len(s3_keys)
await claim_job(worker_id="w-pool")

counter = [0]
def counting_process(keys):
    counter[0] += len(keys)

pool = JobWorkerPool(job.job_id, workers=3, batch_processor=counting_process)
await pool.start()
await pool.join()
print(f"pool processed {counter[0]} photos, fail={job.failed}  state={job.state.value}")


## Idempotent Re-Index via `sha256`

The remaining piece of the durability story is making a full re-run cheap. Without the `sha256` column, re-running the pipeline over the same prefix re-embeds every photo even if its S3 object did not move. With it (introduced in PHT:01), the re-index scan fetches every `s3_key` and its S3 `ETag`, then issues a single SQL batch that asks "which of these `(s3_key, etag)` pairs are already in `photos`?" Only the new ones go to the worker pool.

:::{.callout-note}
ETag is a cheap sufficient dedup signal: S3 assigns a content-derived MD5 as ETag for non-multipart uploads. A changed photo (re-export, edited) has a different ETag even if the `s3_key` is the same, which is what triggers re-index. The sha256 column is the strong dedup; the ETag is the cheap pre-filter.

:::


In [ ]:
PHOTOS_BY_S3_KEY: dict[str, dict] = {}    # s3_key -> photo row with sha256


async def reindex_filter_unchanged(
    s3_key_to_etag: dict[str, str],
) -> tuple[list[str], list[str]]:
    """Returns (to_index, skipped). The to_index list goes to the worker pool.

    In SQL this is a single anti-join:
        SELECT s.s3_key FROM (VALUES (...)) s(s3_key, etag)
        LEFT JOIN photos.photos p USING (s3_key)
        WHERE p.s3_key IS NULL OR p.etag != s.etag
    """
    to_index: list[str] = []
    skipped: list[str] = []
    for s3_key, etag in s3_key_to_etag.items():
        existing = PHOTOS_BY_S3_KEY.get(s3_key)
        if existing and existing.get("etag") == etag:
            skipped.append(s3_key)
        else:
            to_index.append(s3_key)
    return to_index, skipped


# Seed photos: one existing photo with a known etag, three new photos.
PHOTOS_BY_S3_KEY = {
    "photos/aa/abcd.jpg": {"etag": "etag-old-001", "sha256": "a" * 64},
}
new_scan = {
    "photos/aa/abcd.jpg": "etag-old-001",       # unchanged — skip
    "photos/aa/abce.jpg": "etag-new-002",       # new — index
    "photos/aa/abcf.jpg": "etag-new-003",       # new — index
    "photos/aa/abcd.jpg": "etag-old-001",       # duplicate scan entry — de-dup
}
to_index, skipped = await reindex_filter_unchanged(new_scan)
print(f"to index: {len(to_index)}  (expected 2 — only truly new files)")
print(f"skipped:  {len(skipped)}   (expected 1 — unchanged from prior run)")


  ## Resuming on API Startup

The lifespan hook in notebook 12's `main.py` does start-up chores (DB connect, migrations). We extend it with **one query**: any job with `state IN ('running', 'paused')` becomes a fresh worker pool. The paused jobs stay paused (the user explicitly paused). The running ones are readmitted and a new worker pool picks them up at `next_pending_batch` — exactly where the prior process left off.

```python
# In lifespan():
async for job in jobs_in_state(["running"]):
    JobWorkerPool(job_id=job.job_id, workers=2).start()
# paused jobs are not auto-resumed; the user calls POST /jobs/{id}/resume to kick them
```

:::{.callout-important}
The restarted running jobs **must** reconcile the in-flight batch. Specifically, after crash there is always at most one `job_batches` row in state `running` with no `completed_at` — the one the worker was mid-batch on when it died. The lifespan hook marks all such `running` batch rows as `failed` with `error='worker process died'` and resumes from the next batch. The per-photo idempotence of the indexer (dedup by `s3_key` + content checksum) makes re-doing that one batch from scratch safe.

:::


## Schema Additions

This notebook adds the `jobs` and `job_batches` tables shown above. It does not modify `photos` — PHT:01 added the `sha256` column it depends on. The `FOR UPDATE SKIP LOCKED` claim pattern used in production is SQL syntactic sugar over the same atomic UPDATE; the in-memory Python version above does the same guarantee with simpler code.

```sql
-- 0021_pht_jobs.sql (repeated for clarity)
CREATE TYPE job_state AS ENUM ('queued','running','paused','completed','failed','canceled');
CREATE TYPE job_kind  AS ENUM ('reindex_prefix','reindex_device','reindex_photo',
                               'generate_derivatives','recalculate_embeddings');

CREATE TABLE jobs (
  job_id        TEXT PRIMARY KEY,
  kind          job_kind NOT NULL,
  scope_json    JSONB NOT NULL,
  state         job_state NOT NULL DEFAULT 'queued',
  total_items   INT,
  processed     INT NOT NULL DEFAULT 0,
  failed        INT NOT NULL DEFAULT 0,
  started_at    TIMESTAMPTZ,
  completed_at  TIMESTAMPTZ,
  last_error    TEXT
);
CREATE TABLE job_batches (
  batch_id     TEXT PRIMARY KEY,
  job_id       TEXT NOT NULL REFERENCES jobs(job_id),
  seq          INT NOT NULL,
  state        job_state NOT NULL DEFAULT 'queued',
  s3_keys_json JSONB NOT NULL,
  completed_at TIMESTAMPTZ,
  error        TEXT,
  UNIQUE (job_id, seq)
);
```


## Summary

This notebook made the indexing pipeline durable. The work of indexing lives in `job_batches` rows whose progression is committed before the worker moves to the next batch, so a restart resumes from exactly the last committed seq. Job state lives in `jobs` and follows a documented FSM that the user can pause, resume, and cancel. Re-index is cheap because of an ETag anti-join that skips photos whose content has not changed — a feature made possible by PHT:01's `sha256` column and an ETag the pipeline already reads. The worker pool is bounded and crashes safely; lifespan startup reclaims any orphaned running jobs and reconciles the one mid-flight batch.

**What changed relative to notebook 12.** The pipeline goes from "in-process `BackgroundTask` with WebSocket broadcast of in-memory counters" to "row-state in Postgres with WebSocket broadcast of `SELECT` results." Re-index goes from "always re-embeds everything" to "skips everything that has not changed."

**What this enables for the rest of PHT.** PHT:07 instruments the worker pool — each stage (scan, claim, embed, embed-cache hit ratio, batch commit, broadcast) becomes a Prometheus metric. PHT:06's delete cascade drops in a new `job_kind='reindex_photo'` job slot when a user deletes one photo (to regenerate that photo's derivatives only if there are duplicates in another role — edge case we will hand). PHT:07's horizontal scaling section reuses the same job tables with `SKIP LOCKED` and one extra column (`claimed_by`) for multi-worker.

---



---


■
